# 文字生圖：prompts.json → 可轉 3D 的透明 PNG（Kaggle 2×T4）

把 `prompts.json` 放進 Drive 的一個資料夾，共用為「知道連結的任何人」可**檢視**，連結貼到區塊 2 的 `IN_FOLDER_URL`。每個項目：

1. **正面**：FLUX.1-schnell（Apache-2.0）依描述生成，自動加上寫實風格、單一物體、中灰背景、平光、正面等條件（character 另加 A-pose、光頭、不拿武器）。
2. **其他視角**（`views` 裡的 left/back/right）：以正面為準用 MV-Adapter 生成，所以是同一個物體。
3. 全部用 isnet 去背，存成透明 PNG：`<name>-front.png`、`<name>-back.png`…，直接放進 3D 流程的 `in/` 就能用。

兩張 T4 各做一半的項目。跑完打包成 `/kaggle/working/imagegen-<時間>.zip`：裡面是生成的圖，加上一份更新過的 `prompts.json`（做完的項目標記 `"done": true`）。**用它取代 Drive 上的 prompts.json，下次就會跳過做完的項目。**

**每次開始前**（Session options）：Accelerator **GPU T4 x2**、**Internet on**。模型 Dataset（`imageto3d-models`）可以 Add Input，MV-Adapter 用的 SDXL 就不用下載；沒加也可以。

```json
{
  "defaults": {"seed": 1000},
  "items": [
    {"name": "weapon-two_hand_brokensword", "prompt": "a chipped steel two-handed greatsword, leather grip, round pommel"},
    {"name": "monster-gelly", "prompt": "a small green jelly slime monster with big eyes", "views": ["front", "back"]},
    {"name": "character-swordsman_female", "prompt": "young female swordsman, steel breastplate, blue tunic", "views": ["front", "left", "back", "right"]}
  ]
}
```

`name` 是 `<類別>-<id>`（類別同 3D 流程）；`views` 省略時只做正面；`prompt` 用英文描述物體本身就好。每個項目也可以寫 `seed`、`style`（預設寫實風格，見 README）。

In [ ]:
# 1. Check GPUs, RAM and disk (Kaggle: Accelerator "GPU T4 x2", Internet on)
import os, time
SESSION_START = time.time()  # Kaggle stops a session 12 hours after it starts.
import platform, shutil, subprocess, sys

print(f"Python: {platform.python_version()}")
# nvidia-smi instead of torch.cuda: this kernel must not hold a CUDA context on GPU 0 for the whole session.
query = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader"],
                       capture_output=True, text=True)
if query.returncode != 0:
    raise RuntimeError("No GPU. Session options > Accelerator > GPU T4 x2.")
GPUS = [line.split(", ") for line in query.stdout.strip().splitlines()]
for index, name, memory, capability in GPUS:
    print(f"GPU {index}: {name}  {memory}  sm_{capability.replace('.', '')}")
GPU_COUNT = len(GPUS)
GPU_CAPABILITY = GPUS[0][3]
if GPU_COUNT < 2:
    print("WARNING: one GPU only (Accelerator > GPU T4 x2 gives two); running one worker.")
torch_version = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"],
                               capture_output=True, text=True).stdout.strip()
print(f"Torch : {torch_version}")
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
print(f"RAM   : {ram_gb:.1f} GB")
print(f"Disk  : {shutil.disk_usage('/').free / 1024**3:.0f} GB free (/), "
      f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.0f} GB free (/kaggle/working)")
nvcc = shutil.which("nvcc")
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1] if nvcc else "nvcc: not found")
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise RuntimeError("No internet. Session options > Internet on (needs a phone-verified account).")

In [ ]:
# 2. Settings, prompts.json and models
# Drive folder shared as "Anyone with the link" (viewer) that holds prompts.json,
# e.g. "https://drive.google.com/drive/folders/<id>". Empty: prompts.json from an attached Kaggle dataset.
IN_FOLDER_URL = ""
STOP_AFTER_HOURS = 11  # no new image starts after this, so results are packed before Kaggle's 12-hour limit
TRANSFORMER_DTYPE = "float16"  # "bfloat16" if FLUX images come out black (NaN)
PIPELINE_REF = "claude/gallant-thompson-py38mf"

if "SESSION_START" not in globals():
    raise RuntimeError("Run block 1 first.")
import glob, os, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

PIPELINE_DIR = "/content/colab-ImageTo3D"
os.makedirs("/content", exist_ok=True)
if not os.path.exists(PIPELINE_DIR):
    sh(f"git clone https://github.com/chousir/colab-ImageTo3D.git {PIPELINE_DIR}")
sh(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", cwd=PIPELINE_DIR)
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
import importlib
import imageto3d.config, imageto3d.models, imageto3d.imagegen
for module in (imageto3d.config, imageto3d.models, imageto3d.imagegen):
    importlib.reload(module)

# prompts.json: the shared Drive folder, else any attached dataset.
sources = []
if IN_FOLDER_URL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown==5.2.0"], check=True)
    import gdown
    shutil.rmtree("/tmp/prompts_in", ignore_errors=True)
    gdown.download_folder(IN_FOLDER_URL, output="/tmp/prompts_in", quiet=True, remaining_ok=True)
    sources.append("/tmp/prompts_in")
sources.append("/kaggle/input")
found = [path for source in sources for path in glob.glob(os.path.join(source, "**", "prompts.json"), recursive=True)]
if not found:
    raise RuntimeError("prompts.json not found: set IN_FOLDER_URL (shared as viewer) or attach a dataset with it.")
ROOT = "/kaggle/working/imagegen"
os.makedirs(ROOT, exist_ok=True)
PROMPTS = os.path.join(ROOT, "prompts.json")
shutil.copy(found[0], PROMPTS)
data = imageto3d.imagegen.load_prompts(PROMPTS)  # checks names, prompts and views
todo = [item for item in data["items"] if not item.get("done")]
print(f"prompts.json from {found[0]}: {len(data['items'])} item(s), {len(todo)} to make")
for item in todo:
    print(f"  {item['name']}: {', '.join(item['views'])}")

MODEL_ROOT = imageto3d.models.find_root("/kaggle/input") or ""
print("Model dataset:", MODEL_ROOT or "none (SDXL and MV-Adapter download from Hugging Face)")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
CONFIG = imageto3d.imagegen.ImageGenConfig(
    root=ROOT, prompts_path=PROMPTS, hf_home="/content/hf_cache", rembg_home="/content/rembg_cache",
    model_root=MODEL_ROOT, transformer_dtype=TRANSFORMER_DTYPE,
    deadline=SESSION_START + STOP_AFTER_HOURS * 3600)
os.makedirs(CONFIG.rembg_home, exist_ok=True)
for rembg_model in glob.glob(os.path.join(MODEL_ROOT, "rembg", "*.onnx")) if MODEL_ROOT else []:
    shutil.copy(rembg_model, CONFIG.rembg_home)
CONFIG_PATH = "/content/imagegen_config.json"
CONFIG.save(CONFIG_PATH)
print("Settings ready.")

In [ ]:
# 3. Install and download FLUX.1-schnell (do not stop it while it runs)
if "CONFIG_PATH" not in globals():
    raise RuntimeError("Run blocks 1 and 2 first.")
import os, subprocess, sys, time

MV_REPO = "/content/MV-Adapter"
MV_COMMIT = "4277e0018232bac82bb2c103caf0893cedb711be"
MARK = "/content/.imagegen_installed_v1"
CONSTRAINTS = "/content/constraints.txt"
# Same pins as the 3D notebooks, plus bitsandbytes (4-bit FLUX) and sentencepiece (T5 tokenizer).
PINNED = [
    "transformers==4.57.3", "diffusers==0.34.0", "huggingface_hub==0.36.0", "accelerate==1.10.1",
    "peft==0.17.1", "omegaconf==2.3.0", "einops==0.8.1", "trimesh==4.8.3", "onnxruntime==1.22.1",
    "pooch==1.8.2", "jsonschema==4.25.1", "jaxtyping==0.3.2", "typeguard==4.4.4",
    "bitsandbytes==0.46.1", "sentencepiece==0.2.0", "hf_xet",
]
PINNED_NO_DEPS = ["rembg==2.0.67", "pymatting==1.1.14"]
NVDIFFRAST = "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3"

if not os.path.exists(MARK):
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    keep = ("numpy==", "scipy==", "torch==", "torchvision==", "triton==", "numba==", "llvmlite==",
            "pillow==", "opencv-python==", "opencv-python-headless==", "scikit-image==")
    with open(CONSTRAINTS, "w") as file:
        file.write("".join(line + "\n" for line in frozen.splitlines() if line.lower().startswith(keep)))
    if not os.path.exists(MV_REPO):
        sh(f"git clone https://github.com/huanngzh/MV-Adapter.git {MV_REPO}")
    sh(f"git checkout -q {MV_COMMIT}", cwd=MV_REPO)
    sh(f"pip install -c {CONSTRAINTS} " + " ".join(f"'{package}'" for package in PINNED))
    sh(f"pip install -c {CONSTRAINTS} --no-deps " + " ".join(PINNED_NO_DEPS))
    sh(f"pip install --no-build-isolation --no-deps {NVDIFFRAST}")
    open(MARK, "w").close()

check = ("import sys; sys.path[:0] = [%r, %r]; import torch, bitsandbytes, rembg; "
         "from diffusers import FluxPipeline, BitsAndBytesConfig; "
         "from mvadapter.pipelines.pipeline_mvadapter_i2mv_sdxl import MVAdapterI2MVSDXLPipeline; "
         "from scripts.inference_i2mv_sdxl import run_pipeline; import imageto3d.imagegen; "
         "print('torch', torch.__version__, 'bitsandbytes', bitsandbytes.__version__, 'imports OK')" % (PIPELINE_DIR, MV_REPO))
result = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(result.stdout, result.stderr[-3000:])
if result.returncode != 0:
    os.remove(MARK)
    raise RuntimeError("Import check failed. Send the last 30 lines above for diagnosis.")

# FLUX.1-schnell weights once for both GPUs (the 23 GB single-file copy in the repo root is skipped).
os.environ["HF_HOME"] = CONFIG.hf_home
from huggingface_hub import snapshot_download
started = time.time()
snapshot_download(CONFIG.flux_model, allow_patterns=[
    "model_index.json", "scheduler/*", "text_encoder/*", "text_encoder_2/*", "tokenizer/*", "tokenizer_2/*",
    "transformer/*", "vae/*"])
print(f"FLUX.1-schnell ready in {(time.time() - started) / 60:.1f} min")

In [ ]:
# 4. Generate: front views with FLUX, then the other views with MV-Adapter; one worker per GPU
if "CONFIG_PATH" not in globals():
    raise RuntimeError("Run blocks 1, 2 and 3 first.")
import os, subprocess, sys, threading, time, zipfile
import imageto3d.imagegen

ready = threading.Event()

def stream(index, process):
    for line in process.stdout:
        if "it/s]" in line or "s/it]" in line or "B/s]" in line:
            continue  # progress bars
        if "model ready" in line:
            ready.set()
        print(f"[gpu{index}] {line}", end="", flush=True)
    ready.set()

workers = []
for index in range(GPU_COUNT):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index), PYTHONPATH=os.pathsep.join([PIPELINE_DIR, MV_REPO]),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    stage = f"{sys.executable} -u -m imageto3d.imagegen --config {CONFIG_PATH} --worker {index} --workers {GPU_COUNT}"
    # Front views first, then the other views: separate processes, so the two models never share memory.
    process = subprocess.Popen(f"{stage} --stage front; {stage} --stage views", shell=True, env=env,
                               cwd=PIPELINE_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    threading.Thread(target=stream, args=(index, process), daemon=True).start()
    workers.append(process)
    if index + 1 < GPU_COUNT:
        ready.wait(timeout=20 * 60)  # load one FLUX at a time; loading needs a lot of system RAM
        ready.clear()

try:
    while any(process.poll() is None for process in workers):
        time.sleep(15)
except KeyboardInterrupt:
    print("Stopping workers...", flush=True)
    for process in workers:
        process.terminate()
finally:
    data = imageto3d.imagegen.load_prompts(PROMPTS)
    finished = imageto3d.imagegen.mark_done(CONFIG, data)
    imageto3d.imagegen.save_prompts(data, PROMPTS)
    images = sorted(name for name in os.listdir(CONFIG.input_dir) if name.endswith(".png")) \
        if os.path.isdir(CONFIG.input_dir) else []
    archive = f"/kaggle/working/imagegen-{time.strftime('%Y%m%d-%H%M')}.zip"
    with zipfile.ZipFile(archive, "w") as bundle:
        for name in images:
            bundle.write(os.path.join(CONFIG.input_dir, name), f"in/{name}")
        bundle.write(PROMPTS, "prompts.json")
    left = [item["name"] for item in data["items"] if not item.get("done")]
    print(f"\nDone. {len(finished)} item(s) finished this run, {len(images)} image(s) in {archive}")
    print("Replace prompts.json in your Drive folder with the one in the zip, so finished items are skipped next time.")
    if left:
        print("Not finished:", ", ".join(left))
        for name in left:
            error_file = os.path.join(CONFIG.work_dir, f"{name}.error.txt")
            if os.path.exists(error_file):
                print(f"  {name}: {open(error_file).read().strip()}")

In [ ]:
# 5. Preview (optional): front views with the background shown in grey
import glob, os
from PIL import Image
from IPython.display import display
fronts = sorted(glob.glob(os.path.join(CONFIG.input_dir, "*.png")))[:24]
if fronts:
    tile = 256
    sheet = Image.new("RGB", (tile * min(len(fronts), 6), tile * ((len(fronts) + 5) // 6)), (128, 128, 128))
    for index, path in enumerate(fronts):
        image = Image.open(path).convert("RGBA")
        image.thumbnail((tile, tile))
        sheet.paste(image, ((index % 6) * tile + (tile - image.width) // 2, (index // 6) * tile + (tile - image.height) // 2), image)
    display(sheet)
    print("\n".join(os.path.basename(path) for path in fronts))

## 出問題時

| 狀況 | 做法 |
|---|---|
| 區塊 2：prompts.json not found | 資料夾要共用為「知道連結的任何人」可檢視，連結貼到 `IN_FOLDER_URL`；檔名必須是 `prompts.json` |
| 區塊 2：must be `<category>-<id>` / unknown views | 照錯誤訊息修正 prompts.json 的那一項 |
| FLUX produced NaN（圖全黑） | 區塊 2 改 `TRANSFORMER_DTYPE = "bfloat16"`（較慢） |
| `WARNING: almost nothing left after background removal` | 去背後幾乎沒東西：改描述（例如物體顏色和背景太接近），或換 `seed` |
| 某個視角不理想 | 換那一項的 `seed`，把它的 `done` 拿掉再跑一次 |
| 區塊 3 安裝失敗 | 看最後 30 行錯誤 |